In [4]:
!pip -q install pandas numpy scikit-learn gradio

In [9]:

from google.colab import files

uploaded = files.upload()

Saving ml-latest-small.zip to ml-latest-small.zip


In [10]:

import os
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = "/content/ml-latest-small.zip"
DATA_DIR = "/content/movie_data"
BASE_DIR = os.path.join(DATA_DIR, "ml-latest-small")

# Check that the ZIP file was uploaded
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "Please upload ml-latest-small.zip using Step 2 first."
    )

# Extract the dataset
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(DATA_DIR)

# Load movie information, ratings and tags
movies = pd.read_csv(os.path.join(BASE_DIR, "movies.csv"))
ratings = pd.read_csv(os.path.join(BASE_DIR, "ratings.csv"))
tags = pd.read_csv(os.path.join(BASE_DIR, "tags.csv"))

# Calculate average movie ratings
rating_stats = ratings.groupby("movieId")["rating"].agg(
    average_rating="mean",
    rating_count="count"
).reset_index()

movies = movies.merge(rating_stats, on="movieId", how="left")
movies["average_rating"] = movies["average_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

# Combine movie tags
tag_text = (
    tags.dropna(subset=["tag"])
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index(name="tags_text")
)

movies = movies.merge(tag_text, on="movieId", how="left")
movies["tags_text"] = movies["tags_text"].fillna("")

# Prepare titles and years
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$")
movies["display_title"] = movies["title"].str.replace(
    r"\s*\(\d{4}\)\s*$", "", regex=True
)

# Optional plot descriptions are not supplied in this dataset
movies["description"] = ""

# Prepare content for recommendations
movies["content"] = (
    movies["genres"].fillna("").str.replace("|", " ", regex=False)
    + " "
    + movies["tags_text"]
)

movies = movies.reset_index(drop=True)

print("SUCCESS! Movie dataset loaded. 🎬")
print("Total movies:", len(movies))
print("Total ratings:", len(ratings))

display(movies[[
    "display_title", "genres", "average_rating", "rating_count"
]].head(10))

SUCCESS! Movie dataset loaded. 🎬
Total movies: 9742
Total ratings: 100836


,display_title,genres,average_rating,rating_count
0,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,3.920930,215
1,Jumanji,Adventure|Children|Fantasy,3.431818,110
2,Grumpier Old Men,Comedy|Romance,3.259615,52
3,Waiting to Exhale,Comedy|Drama|Romance,2.357143,7
4,Father of the Bride Part II,Comedy,3.071429,49
5,Heat,Action|Crime|Thriller,3.946078,102
6,Sabrina,Comedy|Romance,3.185185,54
7,Tom and Huck,Adventure|Children,2.875000,8
8,Sudden Death,Action,3.125000,16
9,GoldenEye,Action|Adventure|Thriller,3.496212,132


In [11]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Rebuild text features in case Step 4 was skipped
movies["content"] = (
    movies["genres"].fillna("").astype(str).str.replace("|", " ", regex=False)
    + " "
    + movies["description"].fillna("").astype(str)
    + " "
    + movies["tags_text"].fillna("").astype(str)
)

# Avoid empty feature documents
movies["content"] = movies["content"].str.strip()
movies.loc[movies["content"] == "", "content"] = "unknown"

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(movies["content"])

# Normalize ratings to the 0–1 range
max_rating = max(float(movies["average_rating"].max()), 1.0)
movies["rating_score"] = movies["average_rating"] / max_rating

print("Recommendation model trained successfully!")
print("Feature matrix shape:", tfidf_matrix.shape)


def recommend_movies(movie_title, genre_filter="All",
                     min_rating=0.0, top_n=10):
    if not movie_title:
        return pd.DataFrame(columns=[
            "Movie", "Genres", "Rating",
            "Similarity", "Why recommended"
        ])

    matches = movies.index[
        movies["title"].astype(str) == movie_title
    ].tolist()

    if not matches:
        return pd.DataFrame(columns=[
            "Movie", "Genres", "Rating",
            "Similarity", "Why recommended"
        ])

    movie_idx = matches[0]

    # Compare the selected movie with every movie
    similarities = cosine_similarity(
        tfidf_matrix[movie_idx],
        tfidf_matrix
    ).ravel()

    results = movies.copy()
    results["similarity"] = similarities

    # Exclude the selected movie itself
    results = results[results.index != movie_idx]

    if genre_filter != "All":
        results = results[
            results["genres"].fillna("").str.contains(
                rf"(?<!\w){genre_filter}(?!\w)",
                case=False,
                regex=True
            )
        ]

    results = results[
        results["average_rating"] >= float(min_rating)
    ].copy()

    if results.empty:
        return pd.DataFrame(columns=[
            "Movie", "Genres", "Rating",
            "Similarity", "Why recommended"
        ])

    # Content similarity is the dominant factor.
    # Rating contributes only a small ranking adjustment.
    results["final_score"] = (
        0.85 * results["similarity"]
        + 0.15 * results["rating_score"]
    )

    selected_genres = set(
        str(movies.iloc[movie_idx]["genres"]).split("|")
    )

    def explain(row):
        shared = selected_genres.intersection(
            set(str(row["genres"]).split("|"))
        )
        reasons = []

        if shared:
            reasons.append("Shared genres: " + ", ".join(sorted(shared)))

        if row["similarity"] >= 0.25:
            reasons.append("Strong text-content match")
        elif row["similarity"] > 0:
            reasons.append("Some matching content")

        if not reasons:
            reasons.append("Higher overall ranking among filtered results")

        return "; ".join(reasons)

    results["reason"] = results.apply(explain, axis=1)
    results = results.sort_values(
        ["final_score", "average_rating"],
        ascending=False
    ).head(int(top_n))

    output = pd.DataFrame({
        "Movie": results["display_title"],
        "Genres": results["genres"],
        "Rating": results["average_rating"].round(2),
        "Similarity": (results["similarity"] * 100).round(1).astype(str) + "%",
        "Why recommended": results["reason"]
    })

    return output.reset_index(drop=True)


# Test the model
test_movie = movies.iloc[0]["title"]
print("Selected movie:", test_movie)
display(recommend_movies(test_movie, top_n=5))

Recommendation model trained successfully!
Feature matrix shape: (9742, 5528)
Selected movie: Toy Story (1995)


,Movie,Genres,Rating,Similarity,Why recommended
0,"Bug's Life, A",Adventure|Animation|Children|Comedy,3.52,53.0%,"Shared genres: Adventure, Animation, Children,..."
1,Asterix and the Vikings (Astérix et les Vikings),Adventure|Animation|Children|Comedy|Fantasy,5.00,44.5%,"Shared genres: Adventure, Animation, Children,..."
2,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,3.87,44.5%,"Shared genres: Adventure, Animation, Children,..."
3,"Emperor's New Groove, The",Adventure|Animation|Children|Comedy|Fantasy,3.72,44.5%,"Shared genres: Adventure, Animation, Children,..."
4,Presto,Animation|Children|Comedy|Fantasy,5.00,39.1%,"Shared genres: Animation, Children, Comedy, Fa..."


In [12]:

import gradio as gr

all_genres = sorted({
    genre.strip()
    for value in movies["genres"].dropna()
    for genre in str(value).split("|")
    if genre.strip() and genre.strip().lower() != "(no genres listed)"
})

movie_choices = sorted(movies["title"].dropna().astype(str).unique())

def app_recommend(selected_movie, genre, min_rating, number_of_movies):
    if not selected_movie:
        return pd.DataFrame(), "Please select a movie."

    result = recommend_movies(
        movie_title=selected_movie,
        genre_filter=genre,
        min_rating=min_rating,
        top_n=int(number_of_movies)
    )

    if result.empty:
        return result, "No matching movies found. Try a different filter."

    return result, f"Found {len(result)} recommendations for **{selected_movie}**."


with gr.Blocks(theme=gr.themes.Soft(), title="CineMatch Movie Recommender") as demo:
    gr.Markdown("""
    # 🎬 CineMatch
    ### Discover your next favorite movie

    Choose a movie and discover similar titles using genres,
    descriptions, tags and ratings.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            selected_movie = gr.Dropdown(
                choices=movie_choices,
                value=movie_choices[0] if movie_choices else None,
                label="Choose a movie",
                allow_custom_value=False,
                filterable=True
            )

            genre = gr.Dropdown(
                choices=["All"] + all_genres,
                value="All",
                label="Filter by genre"
            )

            min_rating = gr.Slider(
                minimum=0,
                maximum=5,
                value=0,
                step=0.5,
                label="Minimum average rating"
            )

            number_of_movies = gr.Slider(
                minimum=5,
                maximum=20,
                value=10,
                step=1,
                label="Number of recommendations"
            )

            recommend_button = gr.Button(
                "Find Similar Movies 🍿",
                variant="primary"
            )

    status = gr.Markdown()

    results = gr.Dataframe(
        headers=[
            "Movie", "Genres", "Rating",
            "Similarity", "Why recommended"
        ],
        label="Recommended Movies",
        interactive=False,
        wrap=True
    )

    recommend_button.click(
        fn=app_recommend,
        inputs=[
            selected_movie, genre, min_rating, number_of_movies
        ],
        outputs=[results, status]
    )

    # Load initial recommendations when the app opens
    demo.load(
        fn=app_recommend,
        inputs=[
            selected_movie, genre, min_rating, number_of_movies
        ],
        outputs=[results, status]
    )

demo.launch(share=True)

/tmp/ipykernel_717/2399439655.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="CineMatch Movie Recommender") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://83bf74e58e38fe637e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:

import os
import joblib
import json

PROJECT_DIR = "/content/cinematch_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Save the cleaned movie dataset
movies.to_csv(
    os.path.join(PROJECT_DIR, "movies_processed.csv"),
    index=False
)

# Save the trained TF-IDF vectorizer and feature matrix
joblib.dump(
    vectorizer,
    os.path.join(PROJECT_DIR, "tfidf_vectorizer.joblib")
)

joblib.dump(
    tfidf_matrix,
    os.path.join(PROJECT_DIR, "tfidf_matrix.joblib")
)

# Save basic project information
metadata = {
    "project_name": "CineMatch Movie Recommendation App",
    "algorithm": "Content-based filtering",
    "text_features": ["genres", "tags", "description"],
    "similarity": "Cosine similarity",
    "rating_weight": 0.15,
    "content_weight": 0.85
}

with open(os.path.join(PROJECT_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# Create a ZIP archive for downloading
import shutil

zip_path = shutil.make_archive(
    "/content/cinematch_project",
    "zip",
    PROJECT_DIR
)

print("Project files saved!")
print("ZIP file:", zip_path)

Project files saved!
ZIP file: /content/cinematch_project.zip


In [14]:

from google.colab import files
files.download("/content/cinematch_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>